# E2B Example: Webpage Designer Example

Design and host webpages with the Agents SDK and E2B.

See how to use the Agents SDK to generate a webpage, regenerate it in a fresh sandbox, use diffs for a faster edit loop, and then do one more screenshot-guided visual pass.

In [ ]:
from pathlib import Path

from IPython import get_ipython


def find_sdk_root(start: Path) -> Path:
    candidates = []
    for base in (start, *start.parents):
        candidates.append(base)
        candidates.append(base / "SDK")
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "agents").exists():
            return candidate.resolve()
    raise RuntimeError(
        "Could not find the local SDK root from the current notebook working directory"
    )


sdk_root = find_sdk_root(Path.cwd().resolve())
ip = get_ipython()
assert ip is not None, "This install cell must run inside an IPython kernel"
ip.run_line_magic("pip", f'install python-dotenv -e "{sdk_root}[e2b]"')

In [ ]:
import os
import time

from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field

from agents import ModelSettings, Runner
from agents.extensions.sandbox import (
    E2BSandboxClient,
    E2BSandboxClientOptions,
    E2BSandboxType,
)
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig
from agents.sandbox.capabilities import Filesystem, Shell
from agents.sandbox.entries import File
from agents.stream_events import AgentUpdatedStreamEvent, RunItemStreamEvent

load_dotenv(find_dotenv(usecwd=True))

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
PORT = 8765
BRIEF = (
    "Design a polished landing page for E2B, a platform that gives AI "
    "agents secure cloud sandboxes "
    "to run code, browse, and build things safely. Primary CTA: Start "
    "building. Secondary CTA: Read the docs."
)
COMMON_SHELL_NOTES = (
    "When you use exec_command, always pass shell='bash' and never rely on the default shell. "
    "Do not use source or try to activate a virtualenv inside the sandbox; run commands directly. "
    "Ripgrep is not installed in this sandbox, so do not use rg; use grep, "
    "sed, cat, find, curl, and Python instead. "
)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY must be set"
assert os.getenv("E2B_API_KEY"), "E2B_API_KEY must be set"


class WebpageDesignResult(BaseModel):
    summary: str
    key_changes: list[str] = Field(min_length=2)
    served_port: int = Field(description="The port where the webpage is served.")


def base_manifest(
    html: str | None = None,
    css: str | None = None,
    review_history: str | None = None,
) -> Manifest:
    html = html or (
        "<!doctype html><html lang='en'><head><meta charset='utf-8'>"
        "<meta name='viewport' content='width=device-width,initial-scale=1'>"
        "<title>E2B</title><link rel='stylesheet' href='styles.css'></head>"
        "<body><main class='page'><section class='hero'><p class='eyebrow'>E2B</p>"
        "<h1>Secure sandboxes for AI agents</h1><p class='lede'>"
        "Start from this tiny shell and turn it into a stronger landing page."
        "</p><div class='actions'><a href='#' class='primary'>Start building</a>"
        "<a href='#' class='secondary'>Read the docs</a></div></section>"
        "</main></body></html>\n"
    )
    css = css or (
        ":root { color-scheme: light; }\n"
        "body { margin: 0; font-family: Georgia, serif; background: #f6f1e8; "
        "color: #111; }\n"
        ".page { min-height: 100vh; display: grid; place-items: center; "
        "padding: 48px 24px; }\n"
        ".hero { max-width: 720px; }\n"
        ".actions { display: flex; gap: 12px; flex-wrap: wrap; }\n"
        ".primary, .secondary { padding: 12px 18px; border-radius: 999px; "
        "text-decoration: none; }\n"
        ".primary { background: #111; color: #fff; }\n"
        ".secondary { border: 1px solid #111; color: #111; }\n"
    )
    review_history = (
        review_history or "# Visual Review History\n\nNo prior screenshot review notes yet.\n"
    )
    return Manifest(
        entries={
            "README.md": File(
                content=(
                    b"Edit only site/index.html and site/styles.css. "
                    b"Serve the site on port 8765 before finishing.\n"
                )
            ),
            "brief.md": File(content=(BRIEF + "\n").encode()),
            "review/history.md": File(content=review_history.encode()),
            "site/index.html": File(content=html.encode()),
            "site/styles.css": File(content=css.encode()),
        }
    )


def build_agent(name: str, *, instructions: str, developer_notes: str) -> SandboxAgent:
    return SandboxAgent(
        name=name,
        model=MODEL,
        instructions=instructions,
        developer_instructions=COMMON_SHELL_NOTES + developer_notes,
        default_manifest=base_manifest(),
        capabilities=[Filesystem(), Shell()],
        model_settings=ModelSettings(tool_choice="required"),
        output_type=WebpageDesignResult,
    )


client = E2BSandboxClient()


async def new_session(manifest: Manifest | None = None) -> object:
    session = await client.create(
        manifest=base_manifest() if manifest is None else manifest,
        options=E2BSandboxClientOptions(
            sandbox_type=E2BSandboxType.E2B,
            exposed_ports=(PORT,),
            allow_internet_access=True,
            pause_on_exit=True,
        ),
    )
    await session.start()
    print(f"sandbox id: {session.state.sandbox_id}")
    return session


async def ensure_preview(session) -> str:
    command = (
        f"if ! curl -fsS http://127.0.0.1:{PORT}/ >/dev/null 2>&1; then "
        f"nohup python3 -m http.server {PORT} --directory site >/tmp/site-server.log 2>&1 & "
        "sleep 2; "
        "fi; "
        f"curl -fsS http://127.0.0.1:{PORT}/ >/dev/null"
    )
    result = await session.exec("bash", "-lc", command, shell=False, timeout=10)
    assert result.ok(), "site is not being served on the preview port yet"
    return (await session.resolve_exposed_port(PORT)).url_for("http")


async def read_site(session) -> tuple[str, str]:
    html_result = await session.exec("bash", "-lc", "cat site/index.html", shell=False, timeout=5)
    css_result = await session.exec("bash", "-lc", "cat site/styles.css", shell=False, timeout=5)
    html = (
        html_result.stdout.decode() if isinstance(html_result.stdout, bytes) else html_result.stdout
    )
    css = css_result.stdout.decode() if isinstance(css_result.stdout, bytes) else css_result.stdout
    return html, css


async def run_builder(*, session, agent, prompt: str, max_turns: int = 10) -> dict:
    def render(value, limit: int = 1200) -> str:
        if value in (None, ""):
            return "(no output)"
        text = value if isinstance(value, str) else repr(value)
        return text if len(text) <= limit else text[:limit] + "\n...[truncated]"

    def render_tool_call(raw_item) -> str:
        if isinstance(raw_item, dict):
            action = raw_item.get("action")
            commands = action.get("commands") if isinstance(action, dict) else None
        else:
            action = getattr(raw_item, "action", None)
            commands = getattr(action, "commands", None)
        if isinstance(commands, list):
            return "\n".join(str(command) for command in commands)
        return render(raw_item)

    started_at = time.perf_counter()
    result = Runner.run_streamed(
        agent,
        prompt,
        max_turns=max_turns,
        run_config=RunConfig(
            sandbox=SandboxRunConfig(session=session),
            tracing_disabled=True,
        ),
    )

    async for event in result.stream_events():
        if isinstance(event, AgentUpdatedStreamEvent):
            print(f"[agent] {event.new_agent.name}")
            continue
        if not isinstance(event, RunItemStreamEvent):
            continue
        if event.name == "tool_called":
            raw_item = event.item.raw_item
            if isinstance(raw_item, dict):
                tool_name = str(raw_item.get("type", "tool_call"))
            else:
                tool_name = str(getattr(raw_item, "type", type(raw_item).__name__))
            print(f"[tool] {tool_name}")
            print(render_tool_call(raw_item))
        elif event.name == "tool_output":
            print("[tool output]")
            print(render(event.item.output))

    elapsed_seconds = round(time.perf_counter() - started_at, 2)
    preview_url = await ensure_preview(session)
    return {
        "sandbox_id": session.state.sandbox_id,
        "preview_url": preview_url,
        "elapsed_seconds": elapsed_seconds,
        "result": result.final_output.model_dump(),
    }

## Step 1: Initial Generation

Build the first complete landing-page draft from the tiny starter manifest. This gives us a baseline preview URL and a concrete first version to compare against later iterations.


In [ ]:
full_build_instructions = (
    "Turn the tiny static site into a polished landing page and return a short structured summary."
)
full_build_notes = (
    "Read brief.md, site/index.html, and site/styles.css before editing. "
    "Edit only site/index.html and site/styles.css. "
    "You may fully rewrite the page if needed. "
    "Keep the result static HTML and CSS, make it bold and mobile-safe, "
    "and include a hero, proof section, feature cards, and a strong CTA row. "
    "Before finishing, make sure site/ is being served on port 8765; "
    "if nothing is listening there yet, start a background server and "
    "verify it with curl. "
    "served_port must be 8765."
)

initial_session = await new_session()
initial_generation = await run_builder(
    session=initial_session,
    agent=build_agent(
        "Initial Webpage Builder",
        instructions=full_build_instructions,
        developer_notes=full_build_notes,
    ),
    prompt=(
        "Take the starter site and build the first landing-page draft. "
        "Include a hero, a short proof section, three feature cards, and "
        "a CTA row. Serve the result on port 8765 before you finish."
    ),
)

print(initial_generation["result"]["summary"])
print(f"preview url: {initial_generation['preview_url']}")
print(f"elapsed: {initial_generation['elapsed_seconds']}s")

## Step 2: Full Regeneration

Start over from the same blank manifest in a fresh sandbox and regenerate the whole page with a different direction. This shows the heavier-weight path when you want a truly new take instead of editing in place and gives us a new preview URL.


In [ ]:
full_regen_session = await new_session()
full_regeneration = await run_builder(
    session=full_regen_session,
    agent=build_agent(
        "Fresh Sandbox Regenerator",
        instructions=full_build_instructions,
        developer_notes=full_build_notes,
    ),
    prompt=(
        "Start from the same tiny starter site, but generate a noticeably "
        "different polished landing page. You can fully rebuild the HTML "
        "and CSS, but keep the same E2B brief and serve the site on port "
        "8765 before you finish."
    ),
)

print(full_regeneration["result"]["summary"])
print(f"preview url: {full_regeneration['preview_url']}")
print(f"elapsed: {full_regeneration['elapsed_seconds']}s")

## Step 3: Diff-Based Refinement

Stay on the original draft and make surgical edits with diffs only. This keeps the same sandbox, files, and preview URL from Step 1.

In [ ]:
diff_refinement = await run_builder(
    session=initial_session,
    agent=build_agent(
        "Diff Refiner",
        instructions=(
            "Refine the existing landing page with small, targeted edits "
            "and return a short structured summary."
        ),
        developer_notes=(
            "Read brief.md, site/index.html, and site/styles.css before editing. "
            "Use apply_patch for focused edits only. "
            "Do not replace the whole page, do not throw away the existing "
            "structure, and keep the changes surgical. "
            "Tighten the hero copy, add a compact proof line, and make "
            "restrained spacing and color improvements. "
            "Before finishing, make sure site/ is being served on port 8765; "
            "if nothing is listening there yet, start a background server "
            "and verify it with curl. "
            "served_port must be 8765."
        ),
    ),
    prompt=(
        "Refine the current landing page with surgical edits only. "
        "Tighten the hero copy, add one compact proof line, and make "
        "restrained spacing and color improvements. Keep the edits small."
    ),
)

print(diff_refinement["result"]["summary"])
print(f"preview url: {diff_refinement['preview_url']}")
print(f"elapsed: {diff_refinement['elapsed_seconds']}s")
same_preview_url = diff_refinement["preview_url"] == initial_generation["preview_url"]
print(f"same url as initial draft: {same_preview_url}")

## Step 4: Screenshot-Guided Warm-Up Pass

This is the first point where we need browser automation, so the screenshot helpers and Playwright install live here instead of at the top of the notebook. We warm one long-lived review sandbox, capture the current preview, and use that screenshot to drive a fresh refinement pass.


In [ ]:
import asyncio
import io
import shlex
from pathlib import Path
from textwrap import dedent

WORKSPACE_ROOT = Path("/workspace")
review_instructions = (
    "Review the provided landing-page screenshot, improve the page based "
    "on what you see, and return a short structured summary."
)
review_notes = (
    "First read review/history.md, brief.md, site/index.html, and site/styles.css. "
    "Then call view_image on review/screenshots/current.png. "
    "Use apply_patch for focused edits only. "
    "Keep the current structure, but improve any obvious visual issues "
    "you can see in the screenshot, especially spacing, hierarchy, "
    "balance, and CTA clarity. "
    "Before finishing, append a short note to review/history.md "
    "describing what you noticed and what you changed this round. "
    "Make sure site/ is still being served on port 8765 and verify it with curl. "
    "served_port must be 8765."
)
review_prompt = (
    "Review review/history.md and the provided screenshot in "
    "review/screenshots/current.png, then make one visual refinement pass."
)


def workspace_path(path: str) -> str:
    return str(WORKSPACE_ROOT / path)


async def read_text_file(session, path: str) -> str:
    handle = await session.read(Path(path))
    return handle.read().decode()


async def write_file(session, path: str, payload: bytes | str) -> None:
    parent = Path(path).parent
    await session.exec(
        "bash", "-lc", f"mkdir -p {shlex.quote(str(parent))}", shell=False, timeout=5
    )
    if isinstance(payload, str):
        payload = payload.encode()
    await session.write(Path(path), io.BytesIO(payload))


async def file_exists(session, path: str) -> bool:
    result = await session.exec(
        "bash", "-lc", f"test -f {shlex.quote(path)}", shell=False, timeout=5
    )
    return result.ok()


async def tail_file(session, path: str, lines: int = 40) -> str:
    result = await session.exec(
        "bash",
        "-lc",
        f"tail -n {lines} {shlex.quote(path)}",
        shell=False,
        timeout=5,
    )
    stdout = result.stdout.decode() if isinstance(result.stdout, bytes) else result.stdout
    stderr = result.stderr.decode() if isinstance(result.stderr, bytes) else result.stderr
    if result.ok():
        return stdout or "(empty file)"
    return stdout or stderr or f"{path} does not exist"


async def ensure_reviewer_browser(session) -> None:
    ready_marker = "review/.playwright-ready"
    if await file_exists(session, ready_marker):
        print("review sandbox browser ready")
        return

    print("installing playwright once in the review sandbox...")
    npm_install_command = (
        "mkdir -p review/playwright review/logs review/screenshots && "
        "cd review/playwright && "
        "if [ ! -x node_modules/.bin/playwright ]; then "
        "npm init -y >/dev/null 2>&1 && "
        "npm install playwright@latest >../logs/playwright-npm-install.log 2>&1; "
        "fi"
    )
    npm_install_result = await session.exec(
        "bash", "-lc", npm_install_command, shell=False, timeout=900
    )
    assert npm_install_result.ok(), await tail_file(
        session, "review/logs/playwright-npm-install.log"
    )

    deps_command = (
        "cd review/playwright && "
        "sudo -n ./node_modules/.bin/playwright install-deps chromium "
        ">../logs/playwright-install-deps.log 2>&1"
    )
    deps_result = await session.exec("bash", "-lc", deps_command, shell=False, timeout=900)
    assert deps_result.ok(), await tail_file(session, "review/logs/playwright-install-deps.log")

    install_command = (
        "cd review/playwright && "
        "PLAYWRIGHT_BROWSERS_PATH=0 ./node_modules/.bin/playwright install chromium "
        ">../logs/playwright-install.log 2>&1"
    )
    install_result = await session.exec("bash", "-lc", install_command, shell=False, timeout=900)
    assert install_result.ok(), await tail_file(session, "review/logs/playwright-install.log")

    capture_script = (
        dedent(
            """
        const { chromium } = require("playwright");
        const fs = require("fs");
        const path = require("path");

        const [url, outputPath] = process.argv.slice(2);

        if (!url || !outputPath) {
          console.error("usage: node capture.js <url> <outputPath>");
          process.exit(2);
        }

        (async () => {
          const browser = await chromium.launch({
            headless: true,
            args: [
              "--disable-dev-shm-usage",
              "--single-process",
              "--no-zygote",
              "--js-flags=--jitless"
            ]
          });
          const context = await browser.newContext({
            ignoreHTTPSErrors: true,
            viewport: { width: 1440, height: 2200 }
          });
          const page = await context.newPage();
          page.setDefaultNavigationTimeout(45000);
          page.setDefaultTimeout(45000);
          await page.goto(url, { waitUntil: "domcontentloaded", timeout: 45000 });
          await page.waitForLoadState("networkidle", { timeout: 5000 }).catch(() => {});
          await page.waitForTimeout(1000);
          fs.mkdirSync(path.dirname(outputPath), { recursive: true });
          await page.screenshot({ path: outputPath, fullPage: true });
          await context.close();
          await browser.close();
        })().catch((error) => {
          console.error(error && error.stack ? error.stack : String(error));
          process.exit(1);
        });
        """
        ).strip()
        + "\n"
    )
    await write_file(session, "review/playwright/capture.js", capture_script)

    ready_result = await session.exec(
        "bash",
        "-lc",
        "mkdir -p review && printf 'ready\\n' > review/.playwright-ready",
        shell=False,
        timeout=5,
    )
    assert ready_result.ok(), "failed to write review sandbox ready marker"
    print("review sandbox browser ready")


async def capture_preview_screenshot(review_session, preview_url: str, output_path: str) -> bytes:
    await ensure_reviewer_browser(review_session)
    log_path = f"review/logs/{Path(output_path).name}.log"
    status_path = f"review/logs/{Path(output_path).name}.status"
    output_abs = workspace_path(output_path)
    log_abs = workspace_path(log_path)
    status_abs = workspace_path(status_path)
    script_abs = workspace_path("review/playwright/capture.js")
    launch_command = (
        f"mkdir -p {shlex.quote(str(Path(output_abs).parent))} "
        f"{shlex.quote(str(Path(log_abs).parent))} && "
        f"rm -f {shlex.quote(output_abs)} {shlex.quote(status_abs)} && "
        f": > {shlex.quote(log_abs)} && "
        f"(PLAYWRIGHT_BROWSERS_PATH=0 node {shlex.quote(script_abs)} "
        f"{shlex.quote(preview_url)} {shlex.quote(output_abs)} "
        f">> {shlex.quote(log_abs)} 2>&1; "
        f"printf '%s\\n' $? > {shlex.quote(status_abs)})"
    )
    await review_session._inner._sandbox.commands.run(
        f"bash -lc {shlex.quote(launch_command)}",
        background=True,
        cwd=workspace_path("review/playwright"),
        timeout=120,
        stdin=False,
    )

    deadline = time.monotonic() + 120
    while time.monotonic() < deadline:
        if await file_exists(review_session, output_path):
            handle = await review_session.read(Path(output_path))
            return handle.read()
        if await file_exists(review_session, status_path):
            break
        await asyncio.sleep(1)

    if await file_exists(review_session, output_path):
        handle = await review_session.read(Path(output_path))
        return handle.read()
    status_text = (
        await read_text_file(review_session, status_path)
        if await file_exists(review_session, status_path)
        else "missing"
    )
    log_tail = (
        await tail_file(review_session, log_path)
        if await file_exists(review_session, log_path)
        else "(log file was not created)"
    )
    raise AssertionError(
        f"{output_path} was not created. status={status_text.strip() or 'empty'}\n{log_tail}"
    )


def extend_review_history(label: str, payload: dict, round_notes: str) -> str:
    key_changes = "\n".join(f"- {item}" for item in payload["result"]["key_changes"])
    return (
        round_notes.rstrip()
        + f"\n\n## Structured summary for screenshot pass {label}\n"
        + f"- Preview URL: {payload['preview_url']}\n"
        + f"- Summary: {payload['result']['summary']}\n"
        + key_changes
        + "\n"
    )


review_session = await new_session()
await ensure_reviewer_browser(review_session)

current_html, current_css = await read_site(initial_session)
current_preview_url = diff_refinement["preview_url"]
review_history = (
    "# Visual Review History\n\n"
    "Start from the diff-refined landing page and keep carrying forward "
    "what each screenshot review noticed.\n\n"
    "## Starting point\n"
    f"- Preview URL: {current_preview_url}\n"
    f"- Summary: {diff_refinement['result']['summary']}\n"
)

warmup_before_path = "review/screenshots/warmup-before.png"
warmup_after_path = "review/screenshots/warmup-after.png"
warmup_before_bytes = await capture_preview_screenshot(
    review_session, current_preview_url, warmup_before_path
)

screenshot_warmup_session = await new_session(
    base_manifest(
        html=current_html,
        css=current_css,
        review_history=review_history,
    )
)
await write_file(screenshot_warmup_session, "review/screenshots/current.png", warmup_before_bytes)
screenshot_warmup = await run_builder(
    session=screenshot_warmup_session,
    agent=build_agent(
        "Screenshot Reviewer Warm-Up",
        instructions=review_instructions,
        developer_notes=review_notes,
    ),
    prompt=review_prompt,
    max_turns=30,
)
await capture_preview_screenshot(
    review_session, screenshot_warmup["preview_url"], warmup_after_path
)
assert await file_exists(review_session, warmup_before_path)
assert await file_exists(review_session, warmup_after_path)
screenshot_warmup["review_sandbox_id"] = review_session.state.sandbox_id
screenshot_warmup["before_screenshot"] = warmup_before_path
screenshot_warmup["after_screenshot"] = warmup_after_path
warmup_notes = await read_text_file(screenshot_warmup_session, "review/history.md")
current_html, current_css = await read_site(screenshot_warmup_session)
current_preview_url = screenshot_warmup["preview_url"]
review_history = extend_review_history("warm-up", screenshot_warmup, warmup_notes)

print("screenshot-guided warm-up")
print(screenshot_warmup["result"]["summary"])
print(f"preview url: {screenshot_warmup['preview_url']}")
print(f"elapsed: {screenshot_warmup['elapsed_seconds']}s")
print(f"review sandbox id: {screenshot_warmup['review_sandbox_id']}")
print(f"before screenshot: {screenshot_warmup['before_screenshot']}")
print(f"after screenshot: {screenshot_warmup['after_screenshot']}")

## Step 5: Screenshot-Guided Multi-Pass Refinement

Now that the review sandbox is warm, run three more screenshot-guided passes. Each pass gets a fresh builder sandbox and a fresh preview URL, while the persistent review sandbox keeps taking screenshots of the current preview so we do not reinstall Playwright every round.


In [ ]:
visual_review_passes = []

for iteration in range(1, 4):
    before_path = f"review/screenshots/pass-{iteration}-before.png"
    after_path = f"review/screenshots/pass-{iteration}-after.png"
    before_bytes = await capture_preview_screenshot(
        review_session, current_preview_url, before_path
    )

    iteration_session = await new_session(
        base_manifest(
            html=current_html,
            css=current_css,
            review_history=review_history,
        )
    )
    await write_file(iteration_session, "review/screenshots/current.png", before_bytes)
    payload = await run_builder(
        session=iteration_session,
        agent=build_agent(
            f"Screenshot Reviewer {iteration}",
            instructions=review_instructions,
            developer_notes=review_notes,
        ),
        prompt=review_prompt,
        max_turns=30,
    )
    await capture_preview_screenshot(review_session, payload["preview_url"], after_path)
    assert await file_exists(review_session, before_path)
    assert await file_exists(review_session, after_path)
    payload["review_sandbox_id"] = review_session.state.sandbox_id
    payload["before_screenshot"] = before_path
    payload["after_screenshot"] = after_path
    round_notes = await read_text_file(iteration_session, "review/history.md")
    current_html, current_css = await read_site(iteration_session)
    current_preview_url = payload["preview_url"]
    review_history = extend_review_history(str(iteration), payload, round_notes)
    visual_review_passes.append(payload)

    print(f"screenshot-guided pass {iteration}")
    print(payload["result"]["summary"])
    print(f"preview url: {payload['preview_url']}")
    print(f"elapsed: {payload['elapsed_seconds']}s")
    print(f"review sandbox id: {payload['review_sandbox_id']}")
    print(f"before screenshot: {payload['before_screenshot']}")
    print(f"after screenshot: {payload['after_screenshot']}")
    print()

## Timing And Preview Summary

Compare the elapsed time for each stage and keep the sandbox IDs and preview URLs together in one place so the live previews are easy to revisit, including the screenshot warm-up and the later multi-pass loop.


In [ ]:
summary_rows = [
    ("initial generation", initial_generation),
    ("full regeneration", full_regeneration),
    ("diff refinement", diff_refinement),
    ("screenshot warm-up", screenshot_warmup),
]
summary_rows.extend(
    (f"screenshot-guided pass {index}", payload)
    for index, payload in enumerate(visual_review_passes, start=1)
)

for label, payload in summary_rows:
    print(label)
    print(f"  sandbox id: {payload['sandbox_id']}")
    print(f"  preview url: {payload['preview_url']}")
    print(f"  elapsed: {payload['elapsed_seconds']}s")
    if "review_sandbox_id" in payload:
        print(f"  review sandbox id: {payload['review_sandbox_id']}")
    if "before_screenshot" in payload:
        print(f"  before screenshot: {payload['before_screenshot']}")
    if "after_screenshot" in payload:
        print(f"  after screenshot: {payload['after_screenshot']}")
    print()